# MSc Thesis
# Mode choice responses during dynamic street space allocation: A stated preference experiment on mode choice in Amsterdam
#### R.F. Ghofir - Delft University of Technology, 2026

## `Part C: Mixed Logit Model Estimation`

### `1. Project setup`

In [1]:
# Biogeme
import biogeme.database as db
from biogeme.expressions import Beta, Variable
from biogeme import models

import sys
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

# Pandas setting to show all columns when displaying a pandas dataframe
pd.set_option('display.max_columns', None)

In [3]:
# Import function from estimator
from mnl_ml_estimator import estimate_mnl, estimate_panel_mnl, create_comparison_table, print_results

### `2. Load dataset`

In [4]:
# Load df_long dataset preprocessed from 01_data_analysis notebook

df_long_file  = Path(f'sp_long_format_recode.csv')
df_long       = pd.read_csv(df_long_file)

### `3. Database and variables`

In [18]:
all_models = {}

In [68]:
# Biogeme database definition
biodata = db.Database('dssa_choice_data', df_long)

# Define variables

# Attributes of car
time_car    = Variable('time_car')
rbt_car     = Variable('rbt_car')
cost_car    = Variable('cost_car')

# Attributes of bus
time_bus    = Variable('time_bus')
rbt_bus     = Variable('rbt_bus')
cost_bus    = Variable('cost_bus')
crowd_bus   = Variable('crowd_bus')

# Attributes of bicycle
time_bike  = Variable('time_bike')
rbt_bike   = Variable('rbt_bike')

# The choice and availabilities of the alternatives
CHOICE      = Variable('CHOICE')
AV1         = Variable('AV1')
AV2         = Variable('AV2')
AV3         = Variable('AV3')

# Socio-economic variables
AGE         = Variable('age_group')
GENDER      = Variable('gender')
EDUCATION   = Variable('education')
EMPLOYMENT  = Variable('employment')
HH_COMP     = Variable('HH_comp')
HH_INCOME   = Variable('income_group')
BIKE        = Variable('bike')
PT_SUBS     = Variable('PT_subs')
ORIGIN      = Variable('home_municipality')
DESTINATION = Variable('work_municipality')

# Travel behaviour variables
TT_CURRENT  = Variable('tb_tt')
TD_CURRENT  = Variable('tb_td')
TC_CURRENT  = Variable('TC_e')
FREQUENCY   = Variable('tb_freq')
SCHEDULE    = Variable('tb_sched')
TR_EARLY    = Variable('tb_tr')
TR_LATE     = Variable('tb_tr_late')

F_SUPPORT   = Variable('FAC1_support')
F_UNCERTAINTY=Variable('FAC2_uncertainty')
F_PREDICT   = Variable('FAC3_reliability')

### `4. ML with Error Components`

In [63]:
# Model name
model_name = 'Nested MXL with additional error-term'
 
# Fixed parameters
B_time_car   = Beta('B_time_car', 0, None, None, 0)
B_time_bus   = Beta('B_time_bus', 0, None, None, 0)
B_time_bike  = Beta('B_time_bike', 0, None, None, 0)
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', 0, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('sigma ups', 1, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Utility functions
V_car = B_time_car * time_car + B_rbt_car * rbt_car + B_cost_car * cost_car
V_bus = ASC_bus + B_time_bus * time_bus + B_rbt_bus * rbt_bus + B_cost_bus * cost_bus + B_crowd_bus * crowd_bus + Ups_switch
V_bike = ASC_bike + B_time_bike * time_bike + B_rbt_bike * rbt_bike + Ups_switch

V = {1: V_car, 2: V_bus,  3: V_bike}

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_mxl function
results_mxl_nested = estimate_mxl(V,AV,CHOICE,biodata,model_name,num_draws = 250)

# Print the results
print_results(results_mxl_nested)

all_models['Linear additive MXL with error component'] = results_mxl_nested

The number of draws (250) is low. The results may not be meaningful.




Number of estimated parameters:	12
Sample size:	4347
Excluded observations:	0
Init log likelihood:	-4661.187
Final log likelihood:	-4013.016
Likelihood ratio test for the init. model:	1296.342
Rho-square for the init. model:	0.139
Rho-square-bar for the init. model:	0.136
Akaike Information Criterion:	8050.032
Bayesian Information Criterion:	8126.559
Final gradient norm:	6.3246E-01
Number of draws:	250
Draws generation time:	0:00:01.624051
Types of draws:	['Ups_switch: NORMAL_HALTON2']
Nbr of threads:	8

              Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike    -1.8647        0.3124        -5.97          0.00
ASC_bus     -1.1474        0.2826        -4.06          0.00
B_cost_bus  -0.1155        0.0257        -4.49          0.00
B_cost_car  -0.1020        0.0323        -3.16          0.00
B_crowd_bus -0.1891        0.0517        -3.66          0.00
B_rbt_bike   0.0197        0.0159         1.24          0.22
B_rbt_bus   -0.0142        0.0141        -1.01          0.31


### `5. Panel ML`

In [69]:
# Biogeme database definition
biodata = db.Database('dssa_choice_data', df_long_recode)
biodata.panel('ID')

obs_per_ind = biodata.data['ID'].value_counts().unique()[0]
print(f'Number of observations per individual: {obs_per_ind}')

# Use biogeme's "generateFlatPanelDataFrame to create a wide database in which each row corresponds to one individual
df_wide = biodata.generate_flat_panel_dataframe(identical_columns=None)

# Rename the columns, such that they run from columnname_{0} to columnname_{n} 
renumbered_columns = {}
for col in df_wide.columns:
    parts = col.split('_')
    if parts[0].isdigit():
        renumbered_columns[col] = (
            f"{'_'.join(parts[1:])}_{int(parts[0])-1}"
        )
    else:
        renumbered_columns[col] = col
        
# Rename the columns using the dictionary
df_wide.rename(columns=renumbered_columns, inplace=True)

# list of df_wide.columns excluding the columns that start with 'Cost'
col_in = df_wide.columns[df_wide.columns.str.startswith('Cost') == False].tolist()
# Convert the columns that do not start with 'Cost' to integers
for col in col_in:
    df_wide[col] = df_wide[col].astype(int)

# Create Biogeme database object
biodata_wide = db.Database('dssa_choice_data_wide', df_wide)

# Show the first rows of the wide database
print(f'The wide dataset has a shape of {biodata_wide.data.shape}')
biodata_wide.data.head()

Number of observations per individual: 9
The wide dataset has a shape of (483, 127)


,age,work_municipality,AV2,FAC3_reliability,HH_income,AV3,FAC1_support,tb_tr_late,TC_e,tb_park,tb_sched,bike,tb_type,home_municipality,tb_td,age_group,HH_comp,PT_subs,tb_freq,education,AV1,tb_tr,block,FAC2_uncertainty,tb_tt,employment,gender,income_group,CHOICE_0,cost_bus_0,cost_car_0,crowd_bus_0,profile_0,rbt_bike_0,rbt_bus_0,rbt_car_0,time_bike_0,time_bus_0,time_car_0,CHOICE_1,cost_bus_1,cost_car_1,crowd_bus_1,profile_1,rbt_bike_1,rbt_bus_1,rbt_car_1,time_bike_1,time_bus_1,time_car_1,CHOICE_2,cost_bus_2,cost_car_2,crowd_bus_2,profile_2,rbt_bike_2,rbt_bus_2,rbt_car_2,time_bike_2,time_bus_2,time_car_2,CHOICE_3,cost_bus_3,cost_car_3,crowd_bus_3,profile_3,rbt_bike_3,rbt_bus_3,rbt_car_3,time_bike_3,time_bus_3,time_car_3,CHOICE_4,cost_bus_4,cost_car_4,crowd_bus_4,profile_4,rbt_bike_4,rbt_bus_4,rbt_car_4,time_bike_4,time_bus_4,time_car_4,CHOICE_5,cost_bus_5,cost_car_5,crowd_bus_5,profile_5,rbt_bike_5,rbt_bus_5,rbt_car_5,time_bike_5,time_bus_5,time_car_5,CHOICE_6,cost_bus_6,cost_car_6,crowd_bus_6,profile_6,rbt_bike_6,rbt_bus_6,rbt_car_6,time_bike_6,time_bus_6,time_car_6,CHOICE_7,cost_bus_7,cost_car_7,crowd_bus_7,profile_7,rbt_bike_7,rbt_bus_7,rbt_car_7,time_bike_7,time_bus_7,time_car_7,CHOICE_8,cost_bus_8,cost_car_8,crowd_bus_8,profile_8,rbt_bike_8,rbt_bus_8,rbt_car_8,time_bike_8,time_bus_8,time_car_8
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0,1,1,0,4,1,0,4,0,0,0,1,0,1,3,5,2,3,4,3,1,3,2,0,30,0,1,0,1,4,2,1,5,5,5,5,25,10,10,2,0,0,2,6,10,10,5,25,0,20,1,4,4,3,2,5,3,10,15,0,10,2,2,0,2,1,3,10,10,15,10,30,2,2,4,3,4,3,3,5,25,-10,30,3,0,4,3,9,10,3,15,5,10,20,3,4,0,2,8,5,10,15,5,-10,10,1,2,2,1,7,3,5,15,5,0,30,2,0,2,1,3,10,5,10,15,-10,20
2,25,1,1,0,4,1,0,4,0,0,1,2,0,0,6,1,0,2,5,1,1,2,2,0,15,0,0,0,3,2,0,2,1,3,10,10,15,10,30,2,4,4,3,2,5,3,10,15,0,10,3,2,2,1,7,3,5,15,5,0,30,2,0,0,2,6,10,10,5,25,0,20,1,4,2,1,5,5,5,5,25,10,10,3,2,4,3,4,3,3,5,25,-10,30,2,0,2,1,3,10,5,10,15,-10,20,1,4,0,2,8,5,10,15,5,-10,10,2,0,4,3,9,10,3,15,5,10,20
3,31,1,1,0,12,1,0,7,7,6,0,2,0,1,19,1,2,0,5,3,1,5,3,0,30,0,1,3,2,2,4,1,9,10,10,5,15,0,10,2,2,0,3,6,10,5,10,5,-10,10,2,0,2,2,5,5,3,10,5,0,30,2,4,4,1,4,3,10,10,5,10,20,1,2,2,2,3,10,3,15,25,10,10,2,0,4,1,2,5,10,15,25,-10,30,2,4,0,3,1,3,5,15,25,0,20,1,0,0,3,8,5,5,5,15,10,30,2,4,2,2,7,3,3,5,15,-10,20
4,49,1,1,1,1,1,-1,4,5,3,0,0,0,1,16,2,3,3,0,1,1,4,1,2,45,0,1,0,1,4,2,3,3,10,10,5,5,0,30,1,0,4,2,4,3,5,15,15,0,10,1,2,2,3,5,5,10,15,15,-10,20,1,4,0,1,6,10,3,15,15,10,30,1,0,2,3,7,3,10,10,25,10,10,1,2,4,2,2,5,5,5,5,10,20,1,4,4,2,9,10,5,10,25,-10,30,1,2,0,1,8,5,3,10,25,0,20,1,0,0,1,1,3,3,5,5,-10,10
5,55,0,1,0,4,1,0,2,1,0,0,1,0,0,13,2,1,0,5,1,1,1,3,0,20,0,0,0,2,0,4,1,2,5,10,15,25,-10,30,2,4,2,2,7,3,3,5,15,-10,20,2,2,4,1,9,10,10,5,15,0,10,1,2,2,2,3,10,3,15,25,10,10,2,0,0,3,8,5,5,5,15,10,30,1,2,0,3,6,10,5,10,5,-10,10,2,4,4,1,4,3,10,10,5,10,20,3,0,2,2,5,5,3,10,5,0,30,1,4,0,3,1,3,5,15,25,0,20


In [70]:
# Choice variable
CHOICE = {q: Variable(f'CHOICE_{q}') for q in range(obs_per_ind)}

# Car
time_car = {q: Variable(f'time_car_{q}') for q in range(obs_per_ind)}
rbt_car = {q: Variable(f'rbt_car_{q}')  for q in range(obs_per_ind)}
cost_car = {q: Variable(f'cost_car_{q}') for q in range(obs_per_ind)}

# Bus
time_bus = {q: Variable(f'time_bus_{q}') for q in range(obs_per_ind)}
rbt_bus = {q: Variable(f'rbt_bus_{q}') for q in range(obs_per_ind)}
cost_bus = {q: Variable(f'cost_bus_{q}') for q in range(obs_per_ind)}
crowd_bus = {q: Variable(f'crowd_bus_{q}') for q in range(obs_per_ind)}

# Bike
time_bike = {q: Variable(f'time_bike_{q}') for q in range(obs_per_ind)}
rbt_bike = {q: Variable(f'rbt_bike_{q}') for q in range(obs_per_ind)}


# Socio-economic variables
AGE         = Variable('age_group')
GENDER      = Variable('gender')
EDUCATION   = Variable('education')
EMPLOYMENT  = Variable('employment')
HH_COMP     = Variable('HH_comp')
HH_INCOME   = Variable('income_group')
BIKE        = Variable('bike')
PT_SUBS     = Variable('PT_subs')
ORIGIN      = Variable('home_municipality')
DESTINATION = Variable('work_municipality')

# Travel behaviour variables
TT_CURRENT  = Variable('tb_tt')
TD_CURRENT  = Variable('tb_td')
TC_CURRENT  = Variable('TC_e')
FREQUENCY   = Variable('tb_freq')
SCHEDULE    = Variable('tb_sched')
TR_EARLY    = Variable('tb_tr')
TR_LATE     = Variable('tb_tr_late')
F_SUPPORT   = Variable('FAC1_support')
F_UNCERTAINTY=Variable('FAC2_uncertainty')
F_PREDICT   = Variable('FAC3_reliability')

#inspect
CHOICE

{0: CHOICE_0,
 1: CHOICE_1,
 2: CHOICE_2,
 3: CHOICE_3,
 4: CHOICE_4,
 5: CHOICE_5,
 6: CHOICE_6,
 7: CHOICE_7,
 8: CHOICE_8}

In [76]:
# Model name
model_name = 'Panel MXL'
 
# Fixed parameters
B_time_car   = Beta('B_time_car', 0, None, None, 0)
B_time_bus   = Beta('B_time_bus', 0, None, None, 0)
B_time_bike  = Beta('B_time_bike', 0, None, None, 0)
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', 0, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)


# Utility functions
V_car = [
    B_time_car * time_car[q]
    + B_rbt_car * rbt_car[q]
    + B_cost_car * cost_car[q]
    for q in range(obs_per_ind)
]

V_bus = [
    ASC_bus
    + B_time_bus * time_bus[q]
    + B_rbt_bus * rbt_bus[q]
    + B_cost_bus * cost_bus[q]
    + B_crowd_bus * crowd_bus[q]
    for q in range(obs_per_ind)
]

V_bike = [
    ASC_bike
    + B_time_bike * time_bike[q]
    + B_rbt_bike * rbt_bike[q]
    for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mnl = estimate_panel_mnl(
    V,
    AV,
    CHOICE,
    obs_per_ind,
    biodata_wide,
    model_name
)

# Print the results
print_results(results_panel_mnl)

all_models['Linear additive Panel MXL'] = results_panel_mnl



Results for model Panel MXL
Nbr of parameters:		11
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-4016.17
Likelihood ratio test (null):		1518.995
Rho square (null):			0.159
Rho bar square (null):			0.157
Akaike Information Criterion:	8054.34
Bayesian Information Criterion:	8100.32

              Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike    -1.4451        0.1624        -8.90          0.00
ASC_bus     -0.8268        0.1396        -5.92          0.00
B_cost_bus  -0.0973        0.0180        -5.39          0.00
B_cost_car  -0.0694        0.0168        -4.13          0.00
B_crowd_bus -0.1546        0.0421        -3.68          0.00
B_rbt_bike   0.0186        0.0125         1.49          0.14
B_rbt_bus   -0.0130        0.0108        -1.21          0.23
B_rbt_car    0.0065        0.0054         1.20          0.23
B_time_bike -0.0490        0.0050        -9.77          0.00
B_time_bus  -0.0434        0.0040       -10.90          0.

### `6. Panel ML with Shared Error Components`

In [67]:
# Model name
model_name = 'Panel MXL with nested error term'
 
# Fixed parameters
B_time_car   = Beta('B_time_car', 0, None, None, 0)
B_time_bus   = Beta('B_time_bus', 0, None, None, 0)
B_time_bike  = Beta('B_time_bike', 0, None, None, 0)
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', 0, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('Ups_sd', 1, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Utility functions
V_car = [
    B_time_car * time_car[q]
    + B_rbt_car * rbt_car[q]
    + B_cost_car * cost_car[q]
    for q in range(obs_per_ind)
]

V_bus = [
    ASC_bus
    + B_time_bus * time_bus[q]
    + B_rbt_bus * rbt_bus[q]
    + B_cost_bus * cost_bus[q]
    + B_crowd_bus * crowd_bus[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V_bike = [
    ASC_bike
    + B_time_bike * time_bike[q]
    + B_rbt_bike * rbt_bike[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_nested = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 250)

# Print the results
print_results(results_panel_mxl_nested)

all_models['Linear additive Panel MXL with error component '] = results_panel_mxl_nested

The number of draws (250) is low. The results may not be meaningful.




Results for model Panel MXL with nested error term
Nbr of parameters:		12
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-3412.458
Likelihood ratio test (null):		2726.42
Rho square (null):			0.285
Rho bar square (null):			0.283
Akaike Information Criterion:	6848.915
Bayesian Information Criterion:	6899.076

              Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike    -2.1880        0.2359        -9.27          0.00
ASC_bus     -1.4418        0.2224        -6.48          0.00
B_cost_bus  -0.1270        0.0233        -5.44          0.00
B_cost_car  -0.1221        0.0284        -4.30          0.00
B_crowd_bus -0.2032        0.0544        -3.74          0.00
B_rbt_bike   0.0197        0.0142         1.39          0.17
B_rbt_bus   -0.0154        0.0139        -1.11          0.27
B_rbt_car    0.0099        0.0090         1.09          0.28
B_time_bike -0.0534        0.0056        -9.46          0.00
B_time_bus  -0.0556        0.0052

### `7. ML with Random Parameters`

In [68]:
# Model name
model_name = 'Panel MXL with random time'
 
# Fixed parameters
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', 0, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)

# Random parameters to model the nest
B_time_car_mean = Beta('B_time_car_mean', -0.05, None, None, 0)
B_time_car_sd   = Beta('B_time_car_sd',    0.02, None, None, 0)
B_time_bus_mean = Beta('B_time_bus_mean', -0.04, None, None, 0)
B_time_bus_sd   = Beta('B_time_bus_sd',    0.02, None, None, 0)
B_time_bike_mean = Beta('B_time_bike_mean', -0.08, None, None, 0)
B_time_bike_sd   = Beta('B_time_bike_sd',    0.02, None, None, 0)

#Define random parameters, normally distributed, designed for monte-carlo simulation
B_time_car_rnd = B_time_car_mean + B_time_car_sd * bioDraws('B_time_car_rnd', 'NORMAL_HALTON2')
B_time_bus_rnd = B_time_bus_mean + B_time_bus_sd * bioDraws('B_time_bus_rnd', 'NORMAL_HALTON2')
B_time_bike_rnd = B_time_bike_mean + B_time_bike_sd * bioDraws('B_time_bike_rnd', 'NORMAL_HALTON2')

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('Ups_sd', 1, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Utility functions
V_car = [
    B_time_car_rnd * time_car[q]
    + B_rbt_car * rbt_car[q]
    + B_cost_car * cost_car[q]
    for q in range(obs_per_ind)
]

V_bus = [
    ASC_bus
    + B_time_bus_rnd * time_bus[q]
    + B_rbt_bus * rbt_bus[q]
    + B_cost_bus * cost_bus[q]
    + B_crowd_bus * crowd_bus[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V_bike = [
    ASC_bike
    + B_time_bike_rnd * time_bike[q]
    + B_rbt_bike * rbt_bike[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_normalBtt = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 250)

# Print the results
print_results(results_panel_mxl_normalBtt)

all_models['Linear additive Panel MXL with normal Btt '] = results_panel_mxl_normalBtt

The number of draws (250) is low. The results may not be meaningful.




Results for model Panel MXL with random time
Nbr of parameters:		15
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-3364.234
Likelihood ratio test (null):		2822.868
Rho square (null):			0.296
Rho bar square (null):			0.292
Akaike Information Criterion:	6758.468
Bayesian Information Criterion:	6821.168

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike         -1.7687        0.2439        -7.25          0.00
ASC_bus          -1.0783        0.1964        -5.49          0.00
B_cost_bus       -0.1229        0.0222        -5.53          0.00
B_cost_car       -0.1353        0.0287        -4.71          0.00
B_crowd_bus      -0.1956        0.0520        -3.76          0.00
B_rbt_bike        0.0212        0.0156         1.36          0.18
B_rbt_bus        -0.0151        0.0132        -1.14          0.25
B_rbt_car         0.0092        0.0091         1.01          0.31
B_time_bike_mean -0.1021        0.0219        -4.65   

In [69]:
# Model name
model_name = 'Panel MXL with random time lognormal'
 
# Fixed parameters
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', 0, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)

# Random parameters to model the nest
B_time_car_mean = Beta('B_time_car_mean', -3, None, None, 0)
B_time_car_sd   = Beta('B_time_car_sd',    1, None, None, 0)
B_time_bus_mean = Beta('B_time_bus_mean', -3, None, None, 0)
B_time_bus_sd   = Beta('B_time_bus_sd',    1, None, None, 0)
B_time_bike_mean = Beta('B_time_bike_mean', -3, None, None, 0)
B_time_bike_sd   = Beta('B_time_bike_sd',    1, None, None, 0)

#Define random parameters, normally distributed, designed for monte-carlo simulation
B_time_car_rnd = -exp(B_time_car_mean + B_time_car_sd * bioDraws('B_time_car_rnd', 'NORMAL_HALTON2'))
B_time_bus_rnd = -exp(B_time_bus_mean + B_time_bus_sd * bioDraws('B_time_bus_rnd', 'NORMAL_HALTON2'))
B_time_bike_rnd = -exp(B_time_bike_mean + B_time_bike_sd * bioDraws('B_time_bike_rnd', 'NORMAL_HALTON2'))

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('Ups_sd', 1, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Utility functions
V_car = [
    B_time_car_rnd * time_car[q]
    + B_rbt_car * rbt_car[q]
    + B_cost_car * cost_car[q]
    for q in range(obs_per_ind)
]

V_bus = [
    ASC_bus
    + B_time_bus_rnd * time_bus[q]
    + B_rbt_bus * rbt_bus[q]
    + B_cost_bus * cost_bus[q]
    + B_crowd_bus * crowd_bus[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V_bike = [
    ASC_bike
    + B_time_bike_rnd * time_bike[q]
    + B_rbt_bike * rbt_bike[q]
    + Ups_switch
    for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_lognormalBtt = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 250)

# Print the results
print_results(results_panel_mxl_lognormalBtt)

all_models['Linear additive Panel MXL with lognormal Btt '] = results_panel_mxl_lognormalBtt

The number of draws (250) is low. The results may not be meaningful.




Results for model Panel MXL with random time lognormal
Nbr of parameters:		15
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-3372.138
Likelihood ratio test (null):		2807.06
Rho square (null):			0.294
Rho bar square (null):			0.291
Akaike Information Criterion:	6774.276
Bayesian Information Criterion:	6836.976

                    Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike          -2.6844        0.2214       -12.13          0.00
ASC_bus           -1.3468        0.2282        -5.90          0.00
B_cost_bus        -0.1511        0.0263        -5.73          0.00
B_cost_car        -0.1623        0.0290        -5.60          0.00
B_crowd_bus       -0.2462        0.0617        -3.99          0.00
B_rbt_bike         0.0197        0.0141         1.40          0.16
B_rbt_bus         -0.0205        0.0157        -1.30          0.19
B_rbt_car          0.0118        0.0090         1.31          0.19
B_time_bike_mean -12.4570        2.8

In [ ]:
# Model name
model_name = 'Panel MXL with random RBT'
 
# Fixed parameters
B_cost_car  = Beta('B_cost_car', 0, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', 0, None, None, 0)
B_time_car   = Beta('B_time_car', 0, None, None, 0)
B_time_bus   = Beta('B_time_bus', 0, None, None, 0)
B_time_bike  = Beta('B_time_bike', 0, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', 0, None, None, 0)
ASC_bus     = Beta('ASC_bus', 0, None, None, 0)
ASC_bike    = Beta('ASC_bike', 0, None, None, 0)

# Random parameters to model the nest
B_rbt_car_mean = Beta('B_rbt_car_mean', 0, None, None, 0)
B_rbt_car_sd   = Beta('B_rbt_car_sd',   0, None, None, 0)
B_rbt_bus_mean = Beta('B_rbt_bus_mean', 0, None, None, 0)
B_rbt_bus_sd   = Beta('B_rbt_bus_sd',    0, None, None, 0)
B_rbt_bike_mean = Beta('B_rbt_bike_mean', 0, None, None, 0)
B_rbt_bike_sd   = Beta('B_rbt_bike_sd',    0, None, None, 0)

#Define random parameters, normally distributed, designed for monte-carlo simulation
B_rbt_car_rnd = B_rbt_car_mean + B_rbt_car_sd * bioDraws('B_rbt_car_rnd', 'NORMAL_HALTON2')
B_rbt_bus_rnd = B_rbt_bus_mean + B_rbt_bus_sd * bioDraws('B_rbt_bus_rnd', 'NORMAL_HALTON2')
B_rbt_bike_rnd = B_rbt_bike_mean + B_rbt_bike_sd * bioDraws('B_rbt_bike_rnd', 'NORMAL_HALTON2')

# Utility functions
V_car = [
    B_rbt_car_rnd * rbt_car[q]
    + B_time_car * time_car[q]
    + B_cost_car * cost_car[q]
    for q in range(obs_per_ind)
]

V_bus = [
    ASC_bus
    + B_rbt_bus_rnd * rbt_bus[q]
    + B_time_bus * time_bus[q]
    + B_cost_bus * cost_bus[q]
    + B_crowd_bus * crowd_bus[q]
    for q in range(obs_per_ind)
]

V_bike = [
    ASC_bike
    + B_rbt_bike_rnd * rbt_bike[q]
    + B_time_bike * time_bike[q]
    for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_normalrbt = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 200)

# Print the results
print_results(results_panel_mxl_normalrbt)

### `8. Final Panel ML `

In [87]:
# Model name
model_name = 'Panel MXL with random time lognormal and sociodemographic interactions'
 
# Fixed parameters
B_cost_car  = Beta('B_cost_car', -0.1623, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', -0.1511, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0.0118, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', -0.0205, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0.0197, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', -0.2462, None, None, 0)
ASC_bus     = Beta('ASC_bus', -1.3468, None, None, 0)
ASC_bike    = Beta('ASC_bike', -2.6844, None, None, 0)

# Random parameters to model the nest
B_time_car_mean = Beta('B_time_car_mean', -3.07, None, None, 0)
B_time_car_sd   = Beta('B_time_car_sd',    0.58, None, None, 0)
B_time_bus_mean = Beta('B_time_bus_mean', -3.45, None, None, 0)
B_time_bus_sd   = Beta('B_time_bus_sd',    1.18, None, None, 0)
B_time_bike_mean = Beta('B_time_bike_mean', -3, None, None, 0)
B_time_bike_sd   = Beta('B_time_bike_sd',    1, None, None, 0)

#Define random parameters, normally distributed, designed for monte-carlo simulation
B_time_car_rnd = -exp(B_time_car_mean + B_time_car_sd * bioDraws('B_time_car_rnd', 'NORMAL_HALTON2'))
B_time_bus_rnd = -exp(B_time_bus_mean + B_time_bus_sd * bioDraws('B_time_bus_rnd', 'NORMAL_HALTON2'))
B_time_bike_rnd = -exp(B_time_bike_mean + B_time_bike_sd * bioDraws('B_time_bike_rnd', 'NORMAL_HALTON2'))

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('sigma_ups', 1, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Interaction effects
B_timecar_int      = Beta('B_timecar_int' , 0, None, None, 0)
#B_time_car_parttime  = Beta('B_time_car_parttime' , 0, None, None, 0)
B_time_car_otherempl = Beta('B_time_car_otherempl' , 0, None, None, 0)
B_time_car_ams_dest = Beta('B_time_car_ams_dest' , 0, None, None, 0)

B_cost_car_inc50_100   = Beta('B_cost_car_inc50_100' , 0, None, None, 0)
B_cost_car_inc100_150   = Beta('B_cost_car_inc100_150' , 0, None, None, 0)
B_cost_car_inc150more   = Beta('B_cost_car_inc150more' , 0, None, None, 0)

B_time_bus_flexsched = Beta('B_time_bus_flexsched' , 0, None, None, 0)

B_cost_bus_inc50_100   = Beta('B_cost_bus_inc50_100' , 0, None, None, 0)
B_cost_bus_inc100_150   = Beta('B_cost_bus_inc100_150' , 0, None, None, 0)
B_cost_bus_inc150more   = Beta('B_cost_bus_inc150more' , 0, None, None, 0)

B_time_bike_parttime  = Beta('B_time_bike_parttime' , 0, None, None, 0)
#B_time_bike_otherempl = Beta('B_time_bike_otherempl' , 0, None, None, 0)
#B_time_bike_ams_dest = Beta('B_time_bike_ams_dest' , 0, None, None, 0)
#B_time_bike_edu_hbo  = Beta('B_time_bike_edu_hbo' , 0, None, None, 0)

B_rbt_bike_ams     = Beta('B_rbt_bike_ams', 0, None, None, 0)
B_rbtebike_int = Beta('B_rbtbike_int' , 0, None, None, 0)

B_PTsubs    = Beta('B_PTsubs'  , 0, None, None, 0)
B_bikeown   = Beta('B_bikeown'  , 0, None, None, 0)

B_support_bus   = Beta('B_support_bus'  , 0, None, None, 0)
B_uncertainty_bus  = Beta('B_uncertainty_bus'  , 0, None, None, 0)
#B_predict_bus   = Beta('B_predict_bus'  , 0, None, None, 0)

B_support_bike   = Beta('B_support_bike'  , 0, None, None, 0)
B_uncertainty_bike  = Beta('B_uncertainty_bike'  , 0, None, None, 0)
#B_predict_bike   = Beta('B_predict_bike'  , 0, None, None, 0)

PT_subs = (PT_SUBS== 1) + (PT_SUBS == 2) + (PT_SUBS == 3)
bike_user = (BIKE== 1) + (BIKE == 2)

# Utility functions
V_car = [

(
    (
        B_time_car_rnd
        + B_timecar_int * TT_CURRENT
        #+ B_time_car_parttime * (EMPLOYMENT == 1)
        + B_time_car_otherempl * (EMPLOYMENT == 3)
        + B_time_car_ams_dest * (DESTINATION == 1)
    )
    * time_car[q]
    +
    B_rbt_car * rbt_car[q]
    +
    (
        B_cost_car
        + B_cost_car_inc50_100 * (HH_INCOME == 1)
        + B_cost_car_inc100_150 * (HH_INCOME == 2)
        + B_cost_car_inc150more * (HH_INCOME == 3)
    )
    * cost_car[q]
)
for q in range(obs_per_ind)
]

V_bus = [

(
    ASC_bus
    + B_PTsubs * PT_subs
    +
    (
        B_time_bus_rnd
        + B_time_bus_flexsched * (SCHEDULE == 1)
    )
    * time_bus[q]
    +
    B_rbt_bus * rbt_bus[q]
    +
    (
        B_cost_bus
        + B_cost_bus_inc50_100 * (HH_INCOME == 1)
        + B_cost_bus_inc100_150 * (HH_INCOME == 2)
        + B_cost_bus_inc150more * (HH_INCOME == 3)
    )
    * cost_bus[q]
    +
    B_crowd_bus * crowd_bus[q]
    +
    Ups_switch
)
for q in range(obs_per_ind)
]

V_bike = [

(
    ASC_bike
    + B_bikeown * bike_user
    +
    (
        B_time_bike_rnd
        + B_time_bike_parttime * (EMPLOYMENT == 1)
        #+ B_time_bike_otherempl * (EMPLOYMENT == 3)
        #+ B_time_bike_ams_dest * (DESTINATION == 1)
        #+ B_time_bike_edu_hbo * (EDUCATION == 2)
    )
    * time_bike[q]
    +
    (
        B_rbt_bike
        + B_rbtebike_int * TR_EARLY
        + B_rbt_bike_ams * (ORIGIN == 1)
    )
    * rbt_bike[q]
    +
    Ups_switch
)
for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_lognormalBtt_interaction = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 25)

# Print the results
print_results(results_panel_mxl_lognormalBtt_interaction)

all_models['Panel MXL random TT, error terms, and sociodemographic'] = results_panel_mxl_lognormalBtt_interaction

The number of draws (25) is low. The results may not be meaningful.




Results for model Panel MXL with random time lognormal and sociodemographic interactions
Nbr of parameters:		30
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-3242.352
Likelihood ratio test (null):		3066.632
Rho square (null):			0.321
Rho bar square (null):			0.315
Akaike Information Criterion:	6544.703
Bayesian Information Criterion:	6670.104

                        Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike              -3.8419        0.3885        -9.89          0.00
ASC_bus               -2.0202        0.3240        -6.24          0.00
B_PTsubs               1.0873        0.2715         4.00          0.00
B_bikeown              1.5891        0.3561         4.46          0.00
B_cost_bus            -0.3048        0.0459        -6.64          0.00
B_cost_bus_inc100_150  0.2705        0.0674         4.02          0.00
B_cost_bus_inc150more  0.3355        0.0773         4.34          0.00
B_cost_bus_inc50_100   0.1375       

In [71]:
# Model name
model_name = 'Panel MXL with random time lognormal and sociodemographic interactions 2'
 
# Fixed parameters
B_cost_car  = Beta('B_cost_car', -0.1976, None, None, 0)
B_cost_bus  = Beta('B_cost_bus', -0.3048, None, None, 0)
B_rbt_car   = Beta('B_rbt_car', 0.0089, None, None, 0)
B_rbt_bus   = Beta('B_rbt_bus', -0.0174, None, None, 0)
B_rbt_bike  = Beta('B_rbt_bike', 0.1555, None, None, 0)
B_crowd_bus = Beta('B_crowd_bus', -0.2614, None, None, 0)
ASC_bus     = Beta('ASC_bus', -2.0202, None, None, 0)
ASC_bike    = Beta('ASC_bike', -3.8419, None, None, 0)

# Random parameters to model the nest
B_time_car_mean = Beta('B_time_car_mean', -2.6214, None, None, 0)
B_time_car_sd   = Beta('B_time_car_sd',    0.4451, None, None, 0)
B_time_bus_mean = Beta('B_time_bus_mean', -3.4123, None, None, 0)
B_time_bus_sd   = Beta('B_time_bus_sd',    1.1986, None, None, 0)
B_time_bike_mean = Beta('B_time_bike_mean', -7.1575, None, None, 0)
B_time_bike_sd   = Beta('B_time_bike_sd',    4.4303, None, None, 0)

#Define random parameters, normally distributed, designed for monte-carlo simulation
B_time_car_rnd = -exp(B_time_car_mean + B_time_car_sd * bioDraws('B_time_car_rnd', 'NORMAL_HALTON2'))
B_time_bus_rnd = -exp(B_time_bus_mean + B_time_bus_sd * bioDraws('B_time_bus_rnd', 'NORMAL_HALTON2'))
B_time_bike_rnd = -exp(B_time_bike_mean + B_time_bike_sd * bioDraws('B_time_bike_rnd', 'NORMAL_HALTON2'))

# Random parameters to model the nest
Ups_mean = Beta('Ups_mean', 0, None, None,1)
sigma_ups = Beta('sigma_ups', 1.6758, None, None, 0)

Ups_switch = Ups_mean + sigma_ups * bioDraws('Ups_switch', 'NORMAL_HALTON2')

# Interaction effects
#B_timecar_int      = Beta('B_timecar_int' , 0, None, None, 0)
#B_time_car_parttime  = Beta('B_time_car_parttime' , 0, None, None, 0)
#B_time_car_otherempl = Beta('B_time_car_otherempl' , 0, None, None, 0)
#B_time_car_ams_dest = Beta('B_time_car_ams_dest' , 0, None, None, 0)

B_cost_car_inc50_100   = Beta('B_cost_car_inc50_100' , -0.0263, None, None, 0)
B_cost_car_inc100_150   = Beta('B_cost_car_inc100_150' , 0.0707, None, None, 0)
B_cost_car_inc150more   = Beta('B_cost_car_inc150more' ,0.2366, None, None, 0)

#B_time_bus_flexsched = Beta('B_time_bus_flexsched' , 0, None, None, 0)

B_cost_bus_inc50_100   = Beta('B_cost_bus_inc50_100' , 0.1375, None, None, 0)
B_cost_bus_inc100_150   = Beta('B_cost_bus_inc100_150' , 0.2705, None, None, 0)
B_cost_bus_inc150more   = Beta('B_cost_bus_inc150more' , 0.3355, None, None, 0)

#B_time_bike_parttime  = Beta('B_time_bike_parttime' , 0, None, None, 0)
#B_time_bike_otherempl = Beta('B_time_bike_otherempl' , 0, None, None, 0)
#B_time_bike_ams_dest = Beta('B_time_bike_ams_dest' , 0, None, None, 0)
#B_time_bike_edu_hbo  = Beta('B_time_bike_edu_hbo' , 0, None, None, 0)

#B_rbt_bike_ams     = Beta('B_rbt_bike_ams', -0.0866, None, None, 0)
B_rbtebike_int = Beta('B_rbtbike_int' , -0.0309, None, None, 0)

B_PTsubs    = Beta('B_PTsubs'  , 1.0873, None, None, 0)
B_bikeown   = Beta('B_bikeown'  , 1.5891, None, None, 0)

PT_subs = (PT_SUBS== 1) + (PT_SUBS == 2) + (PT_SUBS == 3)
bike_user = (BIKE== 1) + (BIKE == 2)


B_support_bus   = Beta('B_support_bus'  , 0, None, None, 0)
#B_uncertainty_bus  = Beta('B_uncertainty_bus'  , 0, None, None, 0)
#B_predict_bus   = Beta('B_predict_bus'  , 0, None, None, 0)

#B_support_bike   = Beta('B_support_bike'  , 0, None, None, 0)
B_uncertainty_bike  = Beta('B_uncertainty_bike'  , 0, None, None, 0)
#B_predict_bike   = Beta('B_predict_bike'  , 0, None, None, 0)

# Utility functions
V_car = [

(
    (
        B_time_car_rnd
        #+ B_timecar_int * TT_CURRENT
        #+ B_time_car_parttime * (EMPLOYMENT == 1)
        #+ B_time_car_otherempl * (EMPLOYMENT == 3)
        #+ B_time_car_ams_dest * (DESTINATION == 1)
    )
    * time_car[q]
    +
    B_rbt_car * rbt_car[q]
    +
    (
        B_cost_car
        + B_cost_car_inc50_100 * (HH_INCOME == 1)
        + B_cost_car_inc100_150 * (HH_INCOME == 2)
        + B_cost_car_inc150more * (HH_INCOME == 3)
    )
    * cost_car[q]
)
for q in range(obs_per_ind)
]

V_bus = [

(
    ASC_bus
    + B_PTsubs * PT_subs
    +
    (
        B_time_bus_rnd
        #+ B_time_bus_flexsched * (SCHEDULE == 1)
    )
    * time_bus[q]
    +
    B_rbt_bus * rbt_bus[q]
    +
    (
        B_cost_bus
        + B_cost_bus_inc50_100 * (HH_INCOME == 1)
        + B_cost_bus_inc100_150 * (HH_INCOME == 2)
        + B_cost_bus_inc150more * (HH_INCOME == 3)
    )
    * cost_bus[q]
    +
    B_crowd_bus * crowd_bus[q]
    + B_support_bus * F_SUPPORT 
    #+ B_uncertainty_bus * F_UNCERTAINTY 
    #+ B_predict_bus * F_PREDICT
    +
    Ups_switch
)
for q in range(obs_per_ind)
]

V_bike = [

(
    ASC_bike
    + B_bikeown * bike_user
    +
    (
        B_time_bike_rnd
        #+ B_time_bike_parttime * (EMPLOYMENT == 1)
        #+ B_time_bike_otherempl * (EMPLOYMENT == 3)
        #+ B_time_bike_ams_dest * (DESTINATION == 1)
        #+ B_time_bike_edu_hbo * (EDUCATION == 2)
    )
    * time_bike[q]
    +
    (
        B_rbt_bike
        + B_rbtebike_int * TR_EARLY
        #+ B_rbt_bike_ams * (ORIGIN == 1)
    )
    * rbt_bike[q]
    
    #+ B_support_bike * F_SUPPORT
    + B_uncertainty_bike * F_UNCERTAINTY 
    #+ B_predict_bike * F_PREDICT
    +
    Ups_switch
)
for q in range(obs_per_ind)
]

V = [{1: V_car[q], 2: V_bus[q],  3: V_bike[q]} for q in range(obs_per_ind)]

# Create a dictionary to describe the availability conditions of each alternative
AV = {1: AV1, 2: AV2, 3:AV3}

# Estimate the model using the estimate_panel_mxl function
results_panel_mxl_lognormalBtt_interaction2 = estimate_panel_mxl(V,AV,CHOICE,obs_per_ind,biodata_wide,model_name,num_draws = 250)

# Print the results
print_results(results_panel_mxl_lognormalBtt_interaction2)

all_models['Panel MXL random TT, error terms, and sociodemographic 2'] = results_panel_mxl_lognormalBtt_interaction2

The number of draws (250) is low. The results may not be meaningful.




Results for model Panel MXL with random time lognormal and sociodemographic interactions 2
Nbr of parameters:		26
Sample size:			483
Excluded data:			0
Null log likelihood:		-4775.668
Final log likelihood:		-3239.053
Likelihood ratio test (null):		3073.23
Rho square (null):			0.322
Rho bar square (null):			0.316
Akaike Information Criterion:	6530.106
Bayesian Information Criterion:	6638.786

                        Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_bike              -4.1055        0.3847       -10.67          0.00
ASC_bus               -2.0349        0.3162        -6.43          0.00
B_PTsubs               1.0262        0.2793         3.67          0.00
B_bikeown              1.8002        0.3698         4.87          0.00
B_cost_bus            -0.2808        0.0457        -6.14          0.00
B_cost_bus_inc100_150  0.2116        0.0695         3.05          0.00
B_cost_bus_inc150more  0.3184        0.0761         4.18          0.00
B_cost_bus_inc50_100   0.1040      

In [73]:
comparison = create_comparison_table(all_models)

comparison.to_excel(
    'model_comparison.xlsx',
    merge_cells=True
)

comparison


C:\Users\rifanfauzan\AppData\Local\Temp\ipykernel_13376\1688698580.py:63: DeprecationWarning: getEstimatedParameters is deprecated; use get_estimated_parameters instead.
  params = results.getEstimatedParameters()
C:\Users\rifanfauzan\AppData\Local\Temp\ipykernel_13376\1688698580.py:63: DeprecationWarning: getEstimatedParameters is deprecated; use get_estimated_parameters instead.
  params = results.getEstimatedParameters()
C:\Users\rifanfauzan\AppData\Local\Temp\ipykernel_13376\1688698580.py:63: DeprecationWarning: getEstimatedParameters is deprecated; use get_estimated_parameters instead.
  params = results.getEstimatedParameters()
C:\Users\rifanfauzan\AppData\Local\Temp\ipykernel_13376\1688698580.py:63: DeprecationWarning: getEstimatedParameters is deprecated; use get_estimated_parameters instead.
  params = results.getEstimatedParameters()
C:\Users\rifanfauzan\AppData\Local\Temp\ipykernel_13376\1688698580.py:63: DeprecationWarning: getEstimatedParameters is deprecated; use get_esti

Linear-additive RUM MNL          \
                                               Value p-value   
Nbr of parameters                                 11           
Sample size                                     4347           
Null log likelihood                     -4775.667619           
Final log likelihood                    -4016.169973           
Likelihood ratio test (null)             1518.995292           
...                                              ...     ...   
B_timebus_int                                                  
B_timecar_int                                                  
B_uncertainty_bike                                             
B_uncertainty_bus                                              
sigma_ups                                                      

                             Linear-additive MNL without alt-specific parameters  \
                                                                           Value   
Nbr of parameters                                                             6    
Sample size                                                                4347    
Null log likelihood                                                -4775.667619    
Final log likelihood                                               -4023.215105    
Likelihood ratio test (null)                                        1504.905027    
...                                                                         ...    
B_timebus_int                                                                      
B_timecar_int                                                                      
B_uncertainty_bike                                                                 
B_uncertainty_bus                                                                  
sigma_ups                                                                          

                                      \
                             p-value   
Nbr of parameters                      
Sample size                            
Null log likelihood                    
Final log likelihood                   
Likelihood ratio test (null)           
...                              ...   
B_timebus_int                          
B_timecar_int                          
B_uncertainty_bike                     
B_uncertainty_bus                      
sigma_ups                              

                             Linear-additive RUM MNL without RBT and Crowding  \
                                                                        Value   
Nbr of parameters                                                           7   
Sample size                                                              4347   
Null log likelihood                                              -4775.667619   
Final log likelihood                                             -4024.855103   
Likelihood ratio test (null)                                      1501.625032   
...                                                                       ...   
B_timebus_int                                                                   
B_timecar_int                                                                   
B_uncertainty_bike                                                              
B_uncertainty_bus                                                               
sigma_ups                                                                       

                                      \
                             p-value   
Nbr of parameters                      
Sample size                            
Null log likelihood                    
Final log likelihood                   
Likelihood ratio test (null)           
...                              ...   
B_timebus_int                          
B_timecar_int                          
B_uncertainty_bike                     
B_uncertainty_bus                      
sigma_ups                              

       